# Vecchia Lag-1 Budget Reallocation

Question: with the original GEMS target-grid resolution, is it necessary to reuse the full current-time local neighborhood at `t-1`, or is part of that budget more useful as additional current-time spatial neighbors?

The total conditioning budget is fixed:

```text
FullLag baseline:
t   : 8 nearest spatial neighbors
t-1 : same location + 8 same-neighborhood lagged neighbors
total = 8 + 1 + 8 = 17

Realloc_k:
t   : 8 + (8-k) nearest spatial neighbors
t-1 : same location + k same-neighborhood lagged neighbors
total = [8 + (8-k)] + [1 + k] = 17
```

This isolates the budget-allocation question: lagged same-neighborhood information versus extra current-time spatial information.


In [12]:
import os
import sys
import time
import io
import contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.fft
import matplotlib.pyplot as plt

AMAREL_SRC = "/home/jl2815/tco"
LOCAL_SRC = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
_src = AMAREL_SRC if os.path.exists(AMAREL_SRC) else LOCAL_SRC
sys.path.insert(0, _src)

from GEMS_TCO import kernels_vecchia
from GEMS_TCO import orderings as _orderings

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

# Keep the original GEMS target-grid resolution.
DELTA_LAT = 0.044
DELTA_LON = 0.063
T_STEPS = 8

print("DEVICE:", DEVICE)
print("SRC:", _src)
print("Grid resolution:", DELTA_LAT, DELTA_LON)


DEVICE: cpu
SRC: /Users/joonwonlee/Documents/GEMS_TCO-1/src
Grid resolution: 0.044 0.063


## Settings

Defaults use the full spatial range but only a short Monte Carlo run. Use this first to identify promising `k` values, then use the focused section at the bottom for a longer run.


In [13]:
# Full target-domain defaults. For a smoke test, use e.g. (-1, 1) and (123, 126).
LAT_RANGE = (-3.0, 2.0)
LON_RANGE = (121.0, 131.0)
MC_NUM_ITERS = 10            # first-stage full-domain screen
SEED = 42

SMOOTH = 0.5
MM_COND_NUMBER = 100
NHEADS = 0

BASE_T_NEIGHBORS = 8
FULL_LAG1_NEIGHBORS = 8
REALLOC_K_VALUES = [4, 5, 6, 7, 8]  # first-stage screen

# Disable Set C/t-2 by setting daily_stride beyond the number of simulated time steps.
DAILY_STRIDE = T_STEPS + 1

LBFGS_STEPS = 5
LBFGS_EVAL = 20
LBFGS_HIST = 10
INIT_NOISE = 0.7
SUPPRESS_FIT_PRINTS = True

# Optional one-shot Godambe check. Monte Carlo below does not compute Godambe.
RUN_ONE_SHOT_GODAMBE = True
HESSIAN_EPS = 1e-4
SCORE_EPS = 1e-5
H_RIDGE_SCALE = 1e-6
GODAMBE_J_METHOD = "block"          # "block" | "per_unit_centered" | "per_unit_uncentered"
GODAMBE_BLOCK_LAT_WIDTH = 0.50
GODAMBE_BLOCK_LON_WIDTH = 0.50
GODAMBE_BLOCK_TIME_WIDTH = 2.0

TRUE_DICT = {
    "sigmasq": 10.0,
    "range_lat": 0.30,
    "range_lon": 0.40,
    "range_time": 2.0,
    "advec_lat": 0.08,
    "advec_lon": -0.126,
    "nugget": 2.5,
}

def make_budget_realloc_specs(k_values):
    specs = {}
    for k in k_values:
        extra_current = FULL_LAG1_NEIGHBORS - k
        limit_A = BASE_T_NEIGHBORS + extra_current
        limit_B = k
        name = f"Realloc_B{k:02d}_A{limit_A:02d}" if k < FULL_LAG1_NEIGHBORS else "FullLag_B08_A08"
        specs[name] = {
            "lag1_nbs": k,
            "extra_current_nbs": extra_current,
            "limit_A": limit_A,
            "limit_B": limit_B,
            "limit_C": 0,
            "total_conditioning": limit_A + 1 + limit_B,
            "description": f"t: {limit_A}; t-1: same loc + {limit_B}; total fixed",
        }
    return specs

MODEL_SPECS = make_budget_realloc_specs(REALLOC_K_VALUES)
pd.DataFrame(MODEL_SPECS).T


,lag1_nbs,extra_current_nbs,limit_A,limit_B,limit_C,total_conditioning,description
Realloc_B04_A12,4,4,12,4,0,17,t: 12; t-1: same loc + 4; total fixed
Realloc_B05_A11,5,3,11,5,0,17,t: 11; t-1: same loc + 5; total fixed
Realloc_B06_A10,6,2,10,6,0,17,t: 10; t-1: same loc + 6; total fixed
Realloc_B07_A09,7,1,9,7,0,17,t: 9; t-1: same loc + 7; total fixed
FullLag_B08_A08,8,0,8,8,0,17,t: 8; t-1: same loc + 8; total fixed


## Simulation Helpers

Each Monte Carlo iteration uses the same simulated field and the same initialization for all allocation choices.


In [14]:
P_LABELS = ["sigmasq", "range_lat", "range_lon", "range_time", "advec_lat", "advec_lon", "nugget"]
P_COLS = ["sigmasq_est", "range_lat_est", "range_lon_est", "range_t_est", "advec_lat_est", "advec_lon_est", "nugget_est"]
SPATIAL_KEYS = ["sigmasq", "range_lat", "range_lon"]
ADVECTION_KEYS = ["advec_lat", "advec_lon"]

def transform_log_phi_to_physical(p):
    phi1, phi2, phi3, phi4 = (torch.exp(p[i]) for i in range(4))
    rlon = 1.0 / phi2
    return torch.stack([
        phi1 / phi2,
        rlon / torch.sqrt(phi3),
        rlon,
        rlon / torch.sqrt(phi4),
        p[4],
        p[5],
        torch.exp(p[6]),
    ])

def get_covariance_on_grid(lx, ly, lt, params):
    params = torch.clamp(params, min=-15.0, max=15.0)
    phi1, phi2, phi3, phi4 = (torch.exp(params[i]) for i in range(4))
    u_lat = lx - params[4] * lt
    u_lon = ly - params[5] * lt
    dist = torch.sqrt(u_lat.pow(2) * phi3 + u_lon.pow(2) + lt.pow(2) * phi4 + 1e-8)
    return (phi1 / phi2) * torch.exp(-dist * phi2)

def build_target_grid(lat_range, lon_range):
    lat0, lat1 = float(min(lat_range)), float(max(lat_range))
    lon0, lon1 = float(min(lon_range)), float(max(lon_range))
    n_lat = int(np.floor((lat1 - lat0) / DELTA_LAT + 1e-9)) + 1
    n_lon = int(np.floor((lon1 - lon0) / DELTA_LON + 1e-9)) + 1
    lats = lat0 + torch.arange(n_lat, device=DEVICE, dtype=DTYPE) * DELTA_LAT
    lons = lon0 + torch.arange(n_lon, device=DEVICE, dtype=DTYPE) * DELTA_LON
    lats = torch.round(lats * 10000) / 10000
    lons = torch.round(lons * 10000) / 10000
    g_lat, g_lon = torch.meshgrid(lats, lons, indexing="ij")
    grid_coords = torch.stack([g_lat.flatten(), g_lon.flatten()], dim=1)
    return lats, lons, grid_coords

def grid_resolution_report(lats, lons):
    lat_d = torch.diff(lats).detach().cpu().numpy()
    lon_d = torch.diff(lons).detach().cpu().numpy()
    return {
        "lat_min_step": float(lat_d.min()) if len(lat_d) else np.nan,
        "lat_max_step": float(lat_d.max()) if len(lat_d) else np.nan,
        "lon_min_step": float(lon_d.min()) if len(lon_d) else np.nan,
        "lon_max_step": float(lon_d.max()) if len(lon_d) else np.nan,
        "lat_first_last": (float(lats[0]), float(lats[-1])),
        "lon_first_last": (float(lons[0]), float(lons[-1])),
    }

def generate_field_values(n_lat, n_lon, t_steps, params):
    cpu = torch.device("cpu")
    f32 = torch.float32
    px, py, pt = 2 * n_lat, 2 * n_lon, 2 * t_steps
    lx = torch.arange(px, device=cpu, dtype=f32) * DELTA_LAT
    lx[px // 2:] -= px * DELTA_LAT
    ly = torch.arange(py, device=cpu, dtype=f32) * DELTA_LON
    ly[py // 2:] -= py * DELTA_LON
    lt = torch.arange(pt, device=cpu, dtype=f32)
    lt[pt // 2:] -= pt
    params_cpu = params.cpu().float()
    Lx, Ly, Lt = torch.meshgrid(lx, ly, lt, indexing="ij")
    C = get_covariance_on_grid(Lx, Ly, Lt, params_cpu)
    S = torch.fft.fftn(C)
    S.real = torch.clamp(S.real, min=0)
    noise = torch.fft.fftn(torch.randn(px, py, pt, device=cpu, dtype=f32))
    field = torch.fft.ifftn(torch.sqrt(S.real) * noise).real[:n_lat, :n_lon, :t_steps]
    return field.to(device=DEVICE, dtype=DTYPE)

def assemble_reg_map(field, grid_coords, true_params, t_offset=21.0):
    nugget_std = torch.sqrt(torch.exp(true_params[6]))
    n_grid = grid_coords.shape[0]
    field_flat = field.reshape(n_grid, field.shape[-1])
    reg_map = {}
    for t_idx in range(field.shape[-1]):
        key = f"t{t_idx}"
        t_val = float(t_offset + t_idx)
        dummy = torch.zeros(7, device=DEVICE, dtype=DTYPE)
        if t_idx > 0:
            dummy[t_idx - 1] = 1.0
        rows = torch.zeros((n_grid, 11), device=DEVICE, dtype=DTYPE)
        rows[:, :2] = grid_coords
        rows[:, 2] = field_flat[:, t_idx] + torch.randn(n_grid, device=DEVICE, dtype=DTYPE) * nugget_std
        rows[:, 3] = t_val
        rows[:, 4:] = dummy.unsqueeze(0).expand(n_grid, -1)
        reg_map[key] = rows.detach()
    return reg_map

def compute_grid_ordering(grid_coords, mm_cond_number):
    coords_np = grid_coords.detach().cpu().numpy()
    ord_mm = _orderings.maxmin_cpp(coords_np)
    nns = _orderings.find_nns_l2(locs=coords_np[ord_mm], max_nn=mm_cond_number)
    return ord_mm, nns

def true_to_log_params(true_dict):
    phi2 = 1.0 / true_dict["range_lon"]
    phi1 = true_dict["sigmasq"] * phi2
    phi3 = (true_dict["range_lon"] / true_dict["range_lat"]) ** 2
    phi4 = (true_dict["range_lon"] / true_dict["range_time"]) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4), true_dict["advec_lat"], true_dict["advec_lon"], np.log(true_dict["nugget"])]

def backmap_params(out_params):
    p = [x.item() if isinstance(x, torch.Tensor) else float(x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        "sigmasq": np.exp(p[0]) / phi2,
        "range_lat": rlon / phi3 ** 0.5,
        "range_lon": rlon,
        "range_time": rlon / phi4 ** 0.5,
        "advec_lat": p[4],
        "advec_lon": p[5],
        "nugget": np.exp(p[6]),
    }

def rmsre_for_keys(est, true_dict, keys, zero_thresh=0.01):
    vals = []
    for key in keys:
        tv = true_dict[key]
        ev = est[key]
        if abs(tv) >= zero_thresh:
            vals.append(((ev - tv) / abs(tv)) ** 2)
        else:
            vals.append(abs(ev - tv) ** 2)
    return float(np.sqrt(np.mean(vals)))

def relative_se_summary(se_by_key, denom_dict, keys, zero_thresh=0.01):
    vals = []
    for key in keys:
        denom = abs(denom_dict[key])
        if denom >= zero_thresh:
            vals.append((se_by_key[key] / denom) ** 2)
        else:
            vals.append(se_by_key[key] ** 2)
    return float(np.sqrt(np.mean(vals)))

def calculate_metrics(out_params, true_dict):
    est = backmap_params(out_params)
    return {
        "overall_rmsre": rmsre_for_keys(est, true_dict, P_LABELS),
        "spatial_rmsre": rmsre_for_keys(est, true_dict, SPATIAL_KEYS),
        "range_time_re": abs(est["range_time"] - true_dict["range_time"]) / abs(true_dict["range_time"]),
        "advec_rmsre": rmsre_for_keys(est, true_dict, ADVECTION_KEYS),
        "nugget_re": abs(est["nugget"] - true_dict["nugget"]) / abs(true_dict["nugget"]),
        "est": est,
    }

def make_random_init(rng, true_log, init_noise):
    noisy = list(true_log)
    for i in [0, 1, 2, 3, 6]:
        noisy[i] = true_log[i] + rng.uniform(-init_noise, init_noise)
    for i in [4, 5]:
        scale = max(abs(true_log[i]), 0.05)
        noisy[i] = true_log[i] + rng.uniform(-2 * scale, 2 * scale)
    return noisy


## Optional Godambe Helpers

Monte Carlo is the primary comparison. The optional Godambe code uses block/cluster scores rather than per-unit OPG.


In [15]:
def finite_diff_hessian(nll_fn, p, eps=HESSIAN_EPS):
    n = p.shape[0]
    H = torch.zeros(n, n, device=p.device, dtype=p.dtype)
    for i in range(n):
        p_p = p.detach().clone()
        p_m = p.detach().clone()
        p_p[i] += eps
        p_m[i] -= eps
        p_p.requires_grad_(True)
        p_m.requires_grad_(True)
        g_p = torch.autograd.grad(nll_fn(p_p), p_p)[0].detach()
        g_m = torch.autograd.grad(nll_fn(p_m), p_m)[0].detach()
        H[i] = (g_p - g_m) / (2.0 * eps)
    return (H + H.T) / 2.0

def vecchia_per_unit_target_coords(model):
    chunks = []
    if model.Heads_data is not None and model.Heads_data.shape[0] > 0:
        chunks.append(model.Heads_data[:, [0, 1, 3]].to(dtype=DTYPE))
    for X_b in [model.X_A, model.X_AB, model.X_ABC]:
        if X_b is not None and X_b.shape[0] > 0:
            chunks.append(X_b[:, -1, :].to(dtype=DTYPE))
    if not chunks:
        return torch.empty((0, 3), device=DEVICE, dtype=DTYPE)
    return torch.cat(chunks, dim=0)

def make_block_ids(target_coords):
    lat = target_coords[:, 0]
    lon = target_coords[:, 1]
    tim = target_coords[:, 2]
    lat_id = torch.floor((lat - lat.min()) / GODAMBE_BLOCK_LAT_WIDTH).to(torch.long)
    lon_id = torch.floor((lon - lon.min()) / GODAMBE_BLOCK_LON_WIDTH).to(torch.long)
    if GODAMBE_BLOCK_TIME_WIDTH is None or GODAMBE_BLOCK_TIME_WIDTH <= 0:
        time_id = torch.zeros_like(lat_id)
    else:
        time_id = torch.floor((tim - tim.min()) / GODAMBE_BLOCK_TIME_WIDTH).to(torch.long)
    n_lon = int(lon_id.max().item()) + 1 if lon_id.numel() else 1
    n_time = int(time_id.max().item()) + 1 if time_id.numel() else 1
    raw_id = (lat_id * n_lon + lon_id) * n_time + time_id
    _, block_id = torch.unique(raw_id, sorted=True, return_inverse=True)
    return block_id

def score_cov_per_unit_centered(score_mat):
    n_units = score_mat.shape[1]
    score_mean = score_mat.mean(dim=1)
    score_centered = score_mat - score_mean.unsqueeze(1)
    if n_units > 1:
        return score_centered @ score_centered.T / (n_units * (n_units - 1))
    return score_mat @ score_mat.T / max(n_units ** 2, 1)

def score_cov_block_cluster(score_mat, target_coords):
    n_units = score_mat.shape[1]
    scores = score_mat.T.contiguous()
    block_id = make_block_ids(target_coords)
    n_blocks = int(block_id.max().item()) + 1 if block_id.numel() else 0
    block_scores = torch.zeros((n_blocks, scores.shape[1]), device=DEVICE, dtype=DTYPE)
    block_scores.index_add_(0, block_id, scores)
    if n_blocks > 1:
        centered = block_scores - block_scores.mean(dim=0, keepdim=True)
        J = centered.T @ centered * (n_blocks / (n_blocks - 1)) / (n_units ** 2)
    else:
        J = block_scores.T @ block_scores / max(n_units ** 2, 1)
    return J, n_blocks

def compute_vecchia_godambe(model, raw_params):
    p_hat = torch.tensor(raw_params[:7], device=DEVICE, dtype=DTYPE, requires_grad=True)

    def nll(p):
        return model.vecchia_batched_likelihood(p)

    H = finite_diff_hessian(nll, p_hat)
    eig = torch.linalg.eigvalsh(H).detach()
    h_abs_min = torch.clamp(torch.min(torch.abs(eig)), min=1e-12)
    h_cond = float((torch.max(torch.abs(eig)) / h_abs_min).detach().cpu())

    beta_hat = model.get_gls_beta(p_hat).detach()

    def per_unit_losses(p):
        return model.vecchia_per_unit_nll_terms(p, beta_hat)

    cols = []
    for k in range(p_hat.shape[0]):
        pp = p_hat.detach().clone()
        pm = p_hat.detach().clone()
        pp[k] += SCORE_EPS
        pm[k] -= SCORE_EPS
        with torch.no_grad():
            cols.append((per_unit_losses(pp) - per_unit_losses(pm)) / (2.0 * SCORE_EPS))
    score_mat = torch.stack(cols)
    n_units = score_mat.shape[1]
    target_coords = vecchia_per_unit_target_coords(model)
    if target_coords.shape[0] != n_units:
        raise RuntimeError(f"target/score mismatch: targets={target_coords.shape[0]}, scores={n_units}")

    score_mean = score_mat.mean(dim=1)
    p_grad = p_hat.detach().clone().requires_grad_(True)
    profile_grad = torch.autograd.grad(nll(p_grad), p_grad)[0].detach()
    score_grad_diff = profile_grad - score_mean

    J_uncentered = score_mat @ score_mat.T / (n_units ** 2)
    J_centered = score_cov_per_unit_centered(score_mat)
    J_block, n_blocks = score_cov_block_cluster(score_mat, target_coords)

    if GODAMBE_J_METHOD == "block":
        J_main = J_block
    elif GODAMBE_J_METHOD == "per_unit_centered":
        J_main = J_centered
    elif GODAMBE_J_METHOD == "per_unit_uncentered":
        J_main = J_uncentered
    else:
        raise ValueError(f"Unknown GODAMBE_J_METHOD={GODAMBE_J_METHOD!r}")

    eye = torch.eye(H.shape[0], device=DEVICE, dtype=DTYPE)
    h_scale = torch.clamp(torch.mean(torch.abs(torch.diag(H))), min=1.0)
    H_inv = torch.linalg.pinv(H + eye * h_scale * H_RIDGE_SCALE)
    Jac = torch.autograd.functional.jacobian(transform_log_phi_to_physical, p_hat)

    def summarize_J(J):
        G_raw = H_inv @ J @ H_inv
        G_phys = Jac @ G_raw @ Jac.T
        se = torch.sqrt(torch.clamp(torch.diag(G_phys), min=0.0)).detach().cpu().numpy()
        se_by_key = dict(zip(P_LABELS, [float(x) for x in se]))
        return se_by_key, {
            "spatial": relative_se_summary(se_by_key, TRUE_DICT, SPATIAL_KEYS),
            "overall": relative_se_summary(se_by_key, TRUE_DICT, P_LABELS),
            "advec": relative_se_summary(se_by_key, TRUE_DICT, ADVECTION_KEYS),
            "nugget": se_by_key["nugget"] / abs(TRUE_DICT["nugget"]),
        }

    se_main, rel_main = summarize_J(J_main)
    se_block, rel_block = summarize_J(J_block)
    se_centered, rel_centered = summarize_J(J_centered)
    se_uncentered, rel_uncentered = summarize_J(J_uncentered)

    return {
        "gim_j_method": GODAMBE_J_METHOD,
        "gim_n_units": int(n_units),
        "gim_n_blocks": int(n_blocks),
        "gim_h_cond_abs": h_cond,
        "gim_score_mean_max_abs": float(torch.max(torch.abs(score_mean)).detach().cpu()),
        "gim_profile_grad_max_abs": float(torch.max(torch.abs(profile_grad)).detach().cpu()),
        "gim_score_profile_diff_max_abs": float(torch.max(torch.abs(score_grad_diff)).detach().cpu()),
        "gim_spatial_rel_se": rel_main["spatial"],
        "gim_overall_rel_se": rel_main["overall"],
        "gim_advec_rel_se": rel_main["advec"],
        "gim_nugget_rel_se": rel_main["nugget"],
        "gim_spatial_rel_se_block": rel_block["spatial"],
        "gim_spatial_rel_se_perunit_centered": rel_centered["spatial"],
        "gim_spatial_rel_se_uncentered": rel_uncentered["spatial"],
        **{f"gim_se_{k}": v for k, v in se_main.items()},
    }


## Fit And Monte Carlo


In [16]:
def fit_vecchia_spec(model_name, spec, reg_map_ord, nns_grid, initial_vals, compute_godambe=False):
    params = [torch.tensor([val], device=DEVICE, dtype=DTYPE, requires_grad=True) for val in initial_vals]
    model = kernels_vecchia.fit_vecchia_lbfgs(
        smooth=SMOOTH,
        input_map=reg_map_ord,
        nns_map=nns_grid,
        mm_cond_number=MM_COND_NUMBER,
        nheads=NHEADS,
        limit_A=spec["limit_A"],
        limit_B=spec["limit_B"],
        limit_C=spec["limit_C"],
        daily_stride=DAILY_STRIDE,
    )

    t0 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            model.precompute_conditioning_sets()
    else:
        model.precompute_conditioning_sets()
    precompute_s = time.time() - t0

    optimizer = model.set_optimizer(params, lr=1.0, max_iter=LBFGS_EVAL, history_size=LBFGS_HIST)
    t1 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            out, n_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    else:
        out, n_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t1

    metrics = calculate_metrics(out, TRUE_DICT)
    godambe = {}
    gim_s = 0.0
    if compute_godambe:
        t2 = time.time()
        godambe = compute_vecchia_godambe(model, [float(x) for x in out[:7]])
        gim_s = time.time() - t2

    return out, float(out[-1]), int(n_iter), precompute_s, fit_s, gim_s, metrics, godambe

def run_experiment(num_iters=MC_NUM_ITERS, seed=SEED, compute_godambe=False, save_csv=True, csv_name=None):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    true_log = true_to_log_params(TRUE_DICT)
    true_params = torch.tensor(true_log, device=DEVICE, dtype=DTYPE)

    lats_grid, lons_grid, grid_coords = build_target_grid(LAT_RANGE, LON_RANGE)
    n_lat, n_lon = len(lats_grid), len(lons_grid)
    print(f"Grid: {n_lat} x {n_lon} x {T_STEPS} = {n_lat * n_lon * T_STEPS:,} rows")
    print("Actual resolution:", grid_resolution_report(lats_grid, lons_grid))
    print("Model specs:")
    display(pd.DataFrame(MODEL_SPECS).T[["limit_A", "limit_B", "total_conditioning", "description"]])

    ord_grid, nns_grid = compute_grid_ordering(grid_coords, MM_COND_NUMBER)
    print("Ordering done")

    records = []
    for it in range(num_iters):
        print(f"\n=== Iteration {it + 1}/{num_iters} ===")
        initial_vals = make_random_init(rng, true_log, INIT_NOISE)
        field = generate_field_values(n_lat, n_lon, T_STEPS, true_params)
        reg_map = assemble_reg_map(field, grid_coords, true_params)
        del field
        reg_map_ord = {k: v[ord_grid] for k, v in reg_map.items()}

        for model_name, spec in MODEL_SPECS.items():
            try:
                print(f"{model_name}: fitting", end="")
                out, loss, n_iter, pre_s, fit_s, gim_s, metrics, godambe = fit_vecchia_spec(
                    model_name, spec, reg_map_ord, nns_grid, initial_vals, compute_godambe=compute_godambe
                )
                est = metrics.pop("est")
                row = {
                    "iter": it + 1,
                    "model": model_name,
                    "lag1_nbs": spec["lag1_nbs"],
                    "extra_current_nbs": spec["extra_current_nbs"],
                    "limit_A": spec["limit_A"],
                    "limit_B": spec["limit_B"],
                    "total_conditioning": spec["total_conditioning"],
                    "loss": round(loss, 6),
                    "overall_rmsre": round(metrics["overall_rmsre"], 6),
                    "spatial_rmsre": round(metrics["spatial_rmsre"], 6),
                    "range_time_re": round(metrics["range_time_re"], 6),
                    "advec_rmsre": round(metrics["advec_rmsre"], 6),
                    "nugget_re": round(metrics["nugget_re"], 6),
                    "precompute_s": round(pre_s, 3),
                    "fit_s": round(fit_s, 3),
                    "gim_s": round(gim_s, 3),
                    "total_s": round(pre_s + fit_s + gim_s, 3),
                    "fit_iter": n_iter,
                }
                row.update({
                    "sigmasq_est": round(est["sigmasq"], 6),
                    "range_lat_est": round(est["range_lat"], 6),
                    "range_lon_est": round(est["range_lon"], 6),
                    "range_t_est": round(est["range_time"], 6),
                    "advec_lat_est": round(est["advec_lat"], 6),
                    "advec_lon_est": round(est["advec_lon"], 6),
                    "nugget_est": round(est["nugget"], 6),
                })
                row.update({k: round(v, 8) if isinstance(v, float) else v for k, v in godambe.items()})
                records.append(row)
                print(f" | loss={loss:.4f} spatial={row['spatial_rmsre']:.4f} overall={row['overall_rmsre']:.4f} time={row['total_s']:.1f}s")
            except Exception as e:
                print(f" | SKIP {model_name}: {type(e).__name__}: {e}")

    df = pd.DataFrame(records)
    if save_csv and len(df):
        out_dir = Path("log")
        out_dir.mkdir(exist_ok=True)
        out_path = out_dir / (csv_name or "sim_vecchia_lag1_budget_realloc_042926_results.csv")
        df.to_csv(out_path, index=False)
        print("Saved:", out_path)
    return df


## Optional One-Shot Godambe

This is diagnostic only. The main evidence should come from the Monte Carlo section below.


In [17]:
if RUN_ONE_SHOT_GODAMBE:
    df_godambe = run_experiment(
        num_iters=1,
        seed=SEED,
        compute_godambe=True,
        save_csv=True,
        csv_name="sim_vecchia_lag1_budget_realloc_godambe_042926_results.csv",
    )
    display(df_godambe.sort_values("gim_spatial_rel_se"))
else:
    df_godambe = pd.DataFrame()
    print("Skipping one-shot Godambe. Set RUN_ONE_SHOT_GODAMBE=True to run it.")


Grid: 114 x 159 x 8 = 145,008 rows
Actual resolution: {'lat_min_step': 0.043999999999999595, 'lat_max_step': 0.04400000000000004, 'lon_min_step': 0.06299999999998818, 'lon_max_step': 0.0630000000000166, 'lat_first_last': (-3.0, 1.972), 'lon_first_last': (121.0, 130.954)}
Model specs:


,limit_A,limit_B,total_conditioning,description
Realloc_B04_A12,12,4,17,t: 12; t-1: same loc + 4; total fixed
Realloc_B05_A11,11,5,17,t: 11; t-1: same loc + 5; total fixed
Realloc_B06_A10,10,6,17,t: 10; t-1: same loc + 6; total fixed
Realloc_B07_A09,9,7,17,t: 9; t-1: same loc + 7; total fixed
FullLag_B08_A08,8,8,17,t: 8; t-1: same loc + 8; total fixed


Ordering done

=== Iteration 1/1 ===
Realloc_B04_A12: fitting | loss=1.4030 spatial=0.0782 overall=0.0637 time=61.5s
Realloc_B05_A11: fitting | loss=1.4162 spatial=0.1080 overall=0.0887 time=57.0s
Realloc_B06_A10: fitting | loss=1.3951 spatial=0.0767 overall=0.0698 time=62.4s
Realloc_B07_A09: fitting | loss=1.4055 spatial=0.1116 overall=0.0884 time=63.2s
FullLag_B08_A08: fitting | loss=1.4052 spatial=0.0839 overall=0.0772 time=62.8s
Saved: log/sim_vecchia_lag1_budget_realloc_godambe_042926_results.csv


,iter,model,lag1_nbs,extra_current_nbs,limit_A,limit_B,total_conditioning,loss,overall_rmsre,spatial_rmsre,...,gim_spatial_rel_se_block,gim_spatial_rel_se_perunit_centered,gim_spatial_rel_se_uncentered,gim_se_sigmasq,gim_se_range_lat,gim_se_range_lon,gim_se_range_time,gim_se_advec_lat,gim_se_advec_lon,gim_se_nugget
3,1,Realloc_B07_A09,7,1,9,7,17,1.405492,0.088380,0.111579,...,0.034574,0.014427,0.014427,0.294660,0.011312,0.014401,0.081740,0.004801,0.006118,0.041750
4,1,FullLag_B08_A08,8,0,8,8,17,1.405204,0.077161,0.083900,...,0.037126,0.014909,0.014909,0.312347,0.012065,0.015707,0.087558,0.005143,0.006690,0.042589
1,1,Realloc_B05_A11,5,3,11,5,17,1.416200,0.088670,0.107988,...,0.037574,0.015221,0.015221,0.312720,0.012121,0.016124,0.085383,0.004347,0.005505,0.042200
2,1,Realloc_B06_A10,6,2,10,6,17,1.395128,0.069807,0.076724,...,0.037621,0.015744,0.015744,0.323505,0.012440,0.015387,0.088622,0.004593,0.005834,0.039386
0,1,Realloc_B04_A12,4,4,12,4,17,1.403021,0.063691,0.078161,...,0.040330,0.017144,0.017144,0.336742,0.013042,0.017231,0.092344,0.004608,0.005951,0.047861


## Monte Carlo Comparison

This is the primary comparison. All models have the same total conditioning budget.


In [18]:
df_mc = run_experiment(
    num_iters=MC_NUM_ITERS,
    seed=SEED,
    compute_godambe=False,
    save_csv=True,
    csv_name="sim_vecchia_lag1_budget_realloc_mc_042926_results.csv",
)
df_mc.head()


Grid: 114 x 159 x 8 = 145,008 rows
Actual resolution: {'lat_min_step': 0.043999999999999595, 'lat_max_step': 0.04400000000000004, 'lon_min_step': 0.06299999999998818, 'lon_max_step': 0.0630000000000166, 'lat_first_last': (-3.0, 1.972), 'lon_first_last': (121.0, 130.954)}
Model specs:


,limit_A,limit_B,total_conditioning,description
Realloc_B04_A12,12,4,17,t: 12; t-1: same loc + 4; total fixed
Realloc_B05_A11,11,5,17,t: 11; t-1: same loc + 5; total fixed
Realloc_B06_A10,10,6,17,t: 10; t-1: same loc + 6; total fixed
Realloc_B07_A09,9,7,17,t: 9; t-1: same loc + 7; total fixed
FullLag_B08_A08,8,8,17,t: 8; t-1: same loc + 8; total fixed


Ordering done

=== Iteration 1/10 ===
Realloc_B04_A12: fitting | loss=1.4030 spatial=0.0782 overall=0.0637 time=37.2s
Realloc_B05_A11: fitting | loss=1.4162 spatial=0.1080 overall=0.0887 time=31.7s
Realloc_B06_A10: fitting | loss=1.3951 spatial=0.0767 overall=0.0698 time=38.1s
Realloc_B07_A09: fitting | loss=1.4055 spatial=0.1116 overall=0.0884 time=40.8s
FullLag_B08_A08: fitting | loss=1.4052 spatial=0.0839 overall=0.0772 time=37.8s

=== Iteration 2/10 ===
Realloc_B04_A12: fitting | loss=1.3915 spatial=0.0783 overall=0.0645 time=43.8s
Realloc_B05_A11: fitting | loss=1.3939 spatial=0.0702 overall=0.0807 time=45.2s
Realloc_B06_A10: fitting | loss=1.4581 spatial=0.0594 overall=0.0586 time=31.3s
Realloc_B07_A09: fitting | loss=1.3990 spatial=0.0594 overall=0.0531 time=45.9s
FullLag_B08_A08: fitting | loss=1.3991 spatial=0.0609 overall=0.0537 time=40.5s

=== Iteration 3/10 ===
Realloc_B04_A12: fitting | loss=1.3978 spatial=0.0470 overall=0.0467 time=45.1s
Realloc_B05_A11: fitting | loss=1.

KeyboardInterrupt: 

In [ ]:
def p90_p10(x):
    return np.percentile(x, 90) - np.percentile(x, 10)

def summarize_mc(df):
    metric_cols = ["loss", "overall_rmsre", "spatial_rmsre", "range_time_re", "advec_rmsre", "nugget_re", "total_s"]
    rows = []
    for lag1_nbs, sub in df.groupby("lag1_nbs"):
        row = {
            "lag1_nbs": lag1_nbs,
            "extra_current_nbs": int(sub["extra_current_nbs"].iloc[0]),
            "limit_A": int(sub["limit_A"].iloc[0]),
            "limit_B": int(sub["limit_B"].iloc[0]),
            "total_conditioning": int(sub["total_conditioning"].iloc[0]),
            "n": len(sub),
        }
        for col in metric_cols:
            vals = sub[col].dropna().to_numpy()
            row[f"{col}_mean"] = np.mean(vals)
            row[f"{col}_median"] = np.median(vals)
            row[f"{col}_p90_p10"] = p90_p10(vals)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("lag1_nbs").reset_index(drop=True)

mc_summary = summarize_mc(df_mc)
mc_summary.sort_values("spatial_rmsre_mean")


In [ ]:
display(mc_summary)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(mc_summary["lag1_nbs"], mc_summary["spatial_rmsre_mean"], marker="o", label="spatial mean")
axes[0].plot(mc_summary["lag1_nbs"], mc_summary["spatial_rmsre_median"], marker="s", label="spatial median")
axes[0].axvline(FULL_LAG1_NEIGHBORS, color="gray", linestyle="--", linewidth=1, label="full lag")
axes[0].set_title("Spatial parameter recovery")
axes[0].set_xlabel("t-1 same-neighborhood count")
axes[0].set_ylabel("RMSRE")
axes[0].legend()

axes[1].plot(mc_summary["lag1_nbs"], mc_summary["overall_rmsre_mean"], marker="o", label="overall")
axes[1].plot(mc_summary["lag1_nbs"], mc_summary["advec_rmsre_mean"], marker="s", label="advec")
axes[1].plot(mc_summary["lag1_nbs"], mc_summary["nugget_re_mean"], marker="^", label="nugget")
axes[1].axvline(FULL_LAG1_NEIGHBORS, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("Monte Carlo realized error")
axes[1].set_xlabel("t-1 same-neighborhood count")
axes[1].legend()

axes[2].plot(mc_summary["lag1_nbs"], mc_summary["loss_mean"], marker="o")
axes[2].axvline(FULL_LAG1_NEIGHBORS, color="gray", linestyle="--", linewidth=1)
axes[2].set_title("Vecchia loss")
axes[2].set_xlabel("t-1 same-neighborhood count")
axes[2].set_ylabel("loss")

plt.tight_layout()
plt.show()


## Focused Long Run

After the first-stage screen, run a longer comparison on a smaller candidate set. By default this keeps `k=8` as the full-lag baseline and compares it with `k=6,7`. Adjust `FOCUSED_K_VALUES` after seeing the first-stage result.


In [ ]:
RUN_FOCUSED_MC = False
FOCUSED_K_VALUES = [6, 7, 8]
FOCUSED_MC_NUM_ITERS = 100
FOCUSED_SEED = 4242

if RUN_FOCUSED_MC:
    _old_specs = MODEL_SPECS
    try:
        MODEL_SPECS = make_budget_realloc_specs(FOCUSED_K_VALUES)
        df_focused = run_experiment(
            num_iters=FOCUSED_MC_NUM_ITERS,
            seed=FOCUSED_SEED,
            compute_godambe=False,
            save_csv=True,
            csv_name="sim_vecchia_lag1_budget_realloc_focused_mc_042926_results.csv",
        )
        focused_summary = summarize_mc(df_focused)
        display(focused_summary.sort_values("spatial_rmsre_mean"))
    finally:
        MODEL_SPECS = _old_specs
else:
    df_focused = pd.DataFrame()
    focused_summary = pd.DataFrame()
    print("Skipping focused long run. Set RUN_FOCUSED_MC=True after the first-stage screen.")


In [ ]:
if len(focused_summary):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(focused_summary["lag1_nbs"], focused_summary["spatial_rmsre_mean"], marker="o", label="spatial mean")
    axes[0].plot(focused_summary["lag1_nbs"], focused_summary["spatial_rmsre_median"], marker="s", label="spatial median")
    axes[0].axvline(FULL_LAG1_NEIGHBORS, color="gray", linestyle="--", linewidth=1, label="full lag")
    axes[0].set_title("Focused spatial recovery")
    axes[0].set_xlabel("t-1 same-neighborhood count")
    axes[0].legend()

    axes[1].plot(focused_summary["lag1_nbs"], focused_summary["overall_rmsre_mean"], marker="o", label="overall")
    axes[1].plot(focused_summary["lag1_nbs"], focused_summary["advec_rmsre_mean"], marker="s", label="advec")
    axes[1].plot(focused_summary["lag1_nbs"], focused_summary["nugget_re_mean"], marker="^", label="nugget")
    axes[1].axvline(FULL_LAG1_NEIGHBORS, color="gray", linestyle="--", linewidth=1)
    axes[1].set_title("Focused realized error")
    axes[1].set_xlabel("t-1 same-neighborhood count")
    axes[1].legend()

    axes[2].plot(focused_summary["lag1_nbs"], focused_summary["loss_mean"], marker="o")
    axes[2].axvline(FULL_LAG1_NEIGHBORS, color="gray", linestyle="--", linewidth=1)
    axes[2].set_title("Focused Vecchia loss")
    axes[2].set_xlabel("t-1 same-neighborhood count")
    axes[2].set_ylabel("loss")
    plt.tight_layout()
    plt.show()


## Interpretation

- If `k=8` wins, the full same-neighborhood lag structure is useful even under a fixed total budget.
- If `k=4..7` wins, some lagged same-neighborhood slots are better reallocated to extra current-time spatial neighbors.
- Because total conditioning size is fixed at 17, differences are not just due to using more points.
- This design keeps the original `0.044 x 0.063` resolution, so the conclusion is tied to the current target-grid resolution.
